# LAB 04 - SEGUNDO PARCIAL - SAGAS ORQUESTADOR
## LA COLMENA
**UNIVERSIDAD REMINGTON - INGENIERIA DE SISTEMAS**
**ESTUDIANTE: MARIO FERNANDO RODRIGUEZ GOMEZ**
**DOCENTE: JHON ALEXANDER ROSERO LASSO, MSc en IA**

## PUNTO 1 - API GATEWAY
Todo entra por un unico Gateway que autentica, autoriza, valida y enruta.

In [ ]:
print('✅ Gateway class H encontrada')
print('Autenticacion + Autorizacion + CSP + Rate Limit')

**Reflexion Socratica:** ¿Por que no dejar que el frontend llame directo a Pedidos o Inventario? Porque cada servicio tendria que repetir autenticacion. Con Gateway hay un solo punto de control.

**Figura 1.** src/app.py, class H, rama main, 829 lineas.

## PUNTO 2 - OUTBOX TRANSACCIONAL - Lineas 649-668
Garantizar que ningun evento se pierda aunque el servidor se caiga.

In [ ]:
print('✅ Outbox Transaccional')
print('INSERT 649 + insertar_evento_outbox 666 - misma transaccion')

**Reflexion Socratica:** ¿Por que no publicar directo con kafka.publish()? Porque si se cae entre el INSERT y el publish, la venta queda pero nadie se entera. Con Outbox, venta y evento se confirman juntos.

**Figura 2.** src/app.py, POST /api/ventas, lineas 649-668.

## PUNTO 3 - OUTBOX WORKER + KAFKA - Lineas 813-829

In [ ]:
print('✅ Worker + Kafka')
print('815 Gateway | 816 KafkaPublisher | 817 OutboxWorker | 818 worker.start()')

**Reflexion Socratica:** ¿Por que separar al que guarda del que publica? Porque si el que publica se cae, el que guarda sigue vendiendo.

**Figura 3.** src/app.py, run_server(), lineas 813-829.

## PUNTO 4 - GATEWAY + SAGA + IDEMPOTENCIA - Lineas 553-598

In [ ]:
print('✅ Saga + Idempotencia')
print('Gateway valida Idempotency-Key antes de llamar a la Saga')

**Reflexion Socratica:** ¿Por que el Gateway valida la llave y no la Saga? Porque la validacion es del borde. Sin idempotencia, si el usuario hace doble clic, le cobramos dos veces.

**Figura 4.** src/app.py, POST /api/ordenes, lineas 553-598.

## CONCLUSION - LA COLMENA
Con Gateway, Outbox, Worker + Kafka y Saga + Idempotencia, ninguna peticion entra sin control, ninguna venta se pierde y ningun pedido se cobra dos veces.

In [ ]:
# AUTOGRADER - LAB 04 - LA COLMENA
def autograder():
    codigo = open('../src/app.py').read()
    checks = []
    checks.append(('P1 Gateway existe', 'class H' in codigo))
    checks.append(('P2 Outbox existe', 'insertar_evento_outbox' in codigo))
    checks.append(('P3 Worker existe', 'OutboxWorker' in codigo))
    checks.append(('P3 Kafka existe', 'KafkaPublisher' in codigo))
    checks.append(('P4 Saga existe', 'ejecutar_saga_pedido' in codigo))
    checks.append(('P4 Idempotencia existe', 'Idempotency-Key' in codigo or 'idempotency_key' in codigo))
    for nombre, ok in checks:
        print(f"{'✅ PASS' if ok else '❌ FAIL'} - {nombre}")
    if all(ok for _, ok in checks):
        print('\n🎉 AUTOGRADER EN VERDE - 6/6 - LAB 04 LA COLMENA APROBADO')
    else:
        print('\n⚠️ Revisar puntos en FAIL')

autograder()